In [1]:
import os,random,glob,zipfile
import numpy as np
import librosa,soundfile as sf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader



In [2]:
#PARAMETERS
SR=16000
DURATION=5.0
TARGET_LEN=int(SR*DURATION)
N_FFT=512
HOP_LENGTH=256
BATCH_SIZE=8
NUM_EPOCHS=50
LR=5e-5
WEIGHT_DECAY=1e-5
PATIENCE=10
USE_AMP=True
def unzip_all(zip_folder, extract_to):
    for file in os.listdir(zip_folder):
        if file.endswith(".zip"):
            path = os.path.join(zip_folder, file)
            print("Extracting:", file)
            with zipfile.ZipFile(path, 'r') as z:
                z.extractall(extract_to)
unzip_all(r"C:\Users\JOSHITAA\Documents\404FunNotFound\AI-Voice-Enhancer\data\noisy datset", r"C:\Users\JOSHITAA\Documents\404FunNotFound\AI-Voice-Enhancer\data\noisy datset")
CLEAN_DATASET_DIR = r"C:\Users\JOSHITAA\Documents\404FunNotFound\AI-Voice-Enhancer\data\clean datset"
NOISE_DATASET_DIR = r"C:\Users\JOSHITAA\Documents\404FunNotFound\AI-Voice-Enhancer\data\noisy datset"
mic_inner = os.path.join(CLEAN_DATASET_DIR, "MIC")
CLEAN_ROOT = mic_inner if os.path.exists(mic_inner) else CLEAN_DATASET_DIR
speaker_folders=sorted([
    d for d in os.listdir(CLEAN_ROOT)
    if os.path.isdir(os.path.join(CLEAN_ROOT, d))
])
print("Speakers found:", speaker_folders)
random.seed(42)
random.shuffle(speaker_folders)
split=int(0.8*len(speaker_folders))
train_speakers=speaker_folders[:split]
val_speakers=speaker_folders[split:]

train_clean_files,val_clean_files=[],[]
for spk in speaker_folders:
    path=os.path.join(CLEAN_ROOT,spk)
    wavs=sorted(glob.glob(os.path.join(path,"*.wav")))
    if spk in train_speakers:
        train_clean_files.extend(wavs)
    else:
        val_clean_files.extend(wavs)
noise_files = sorted(
    glob.glob(os.path.join(NOISE_DATASET_DIR, "**", "*.wav"), recursive=True)
)
assert len(train_clean_files)>0, "No training clean files found"
assert len(val_clean_files)>0, "validation clean set is empty"
assert len(noise_files)>0, "No noise files found"
print("Noise files found:", len(noise_files))
print("Example noise file:", noise_files[:5])

Extracting: DKITCHEN_16k.zip
Extracting: DLIVING_16k.zip
Extracting: DWASHING_16k.zip
Extracting: NFIELD_16k.zip
Extracting: NPARK_16k.zip
Extracting: NRIVER_16k.zip
Extracting: OHALLWAY_16k.zip
Extracting: OMEETING_16k.zip
Extracting: PRESTO_16k.zip
Extracting: PSTATION_16k.zip
Speakers found: ['F01', 'F02', 'M01', 'M02']
Noise files found: 160
Example noise file: ['C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch01.wav', 'C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch02.wav', 'C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch03.wav', 'C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch04.wav', 'C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch05.wav']


In [3]:
#MIXING FUNCTION
def mix_with_random_noise(clean, noise_files,sr=SR,target_len=TARGET_LEN):
    clean=librosa.util.fix_length(clean,size=target_len)

    noise_path=random.choice(noise_files)
    noise,_=librosa.load(noise_path,sr=sr)

    if len(noise)>target_len:
        start=random.radint(0,len(noise)-target_len)
        noise=noise[start:start+target_len]
    else:
        noise=np.pad(noise,(0,target_len-len(noise)))

    if random.random()<0.25:
        try:
            speed = random.uniform(0.9, 1.1)
            noise = librosa.effects.time_stretch(noise, rate=speed)
            noise = librosa.util.fix_length(noise, size=target_len)
        except Exception:
            pass


    snr_db = random.uniform(0, 20)
    rms_clean = np.sqrt(np.mean(clean**2) + 1e-12)
    rms_noise = np.sqrt(np.mean(noise**2) + 1e-12)
    desired_rms_noise = rms_clean / (10 ** (snr_db / 20))
    noise = noise * (desired_rms_noise / (rms_noise + 1e-12))
    noise = noise * random.uniform(0.7, 1.2)

    mixed = clean + noise

    maxv = np.max(np.abs(mixed)) + 1e-12
    if maxv > 1.0:
        mixed = mixed / maxv
    return mixed.astype(np.float32), clean.astype(np.float32)

In [4]:
#DATASET
class SpeechMaskDataset(Dataset):
    def __init__(self,clean_files,noise_files,sr=SR,n_fft=N_FFT,hop_length=HOP_LENGTH,augment=True):
        self.clean_files=clean_files
        self.noise_files=noise_files
        self.sr=sr
        self.n_fft=n_fft
        self.hop_length=hop_length
        self.augment=augment

    def __len__(self):
        return len(self.clean_files)
    def __getitem__(self,idx):
        clean_wav, _=librosa.load(self.clean_files[idx],sr=self.sr)
        mixed_wav,clean_wav=mix_with_random_noise(clean_wav,self.noise_files,sr=self.sr)

        noisy_stft=librosa.stft(mixed_wav,n_fft=self.n_fft,hop_length=self.hop_length)
        clean_stft=librosa.stft(clean_wav,n_fft=self.n_fft,hop_length=self.hop_length)

        noisy_mag=np.abs(noisy_stft)
        clean_mag=np.abs(clean_stft)

        mask=clean_mag/(noisy_mag+1e-8)
        mask=np.click(mask,0.0,1.0)

        noisy_mag=np.log1p(noisy_mag)
        noisy_mag_norm=noisy_mag/(noisy_mag.max()+1e-8)

        noisy_tensor=torch.tensor(noisy_mag_norm, dtype=torch.float32)
        mask_tensor=torch.tensor(mask,dtype=torch.float32).unsqueeze(0)

        return noisy_tensor,mask_tensor,mixed_wav.astype(np.float32), clean_wav.astype(np.float32)
train_dataset=SpeechMaskDataset(train_clean_files,noise_files)
val_dataset=SpeechMaskDataset(val_clean_files,noise_files)

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True, drop_last=True)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False, drop_last=False)
print(f"Data ready | Train: {len(train_dataset)} | Val: {len(val_dataset)}")


Data ready | Train: 708 | Val: 236


In [5]:
#MODEL
class UNetMask(nn.Module):
    def __init__(self):
        super().__init__()
        def down(in_ch,out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch,out_ch,3,2,1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )
        def up(in_ch,out_ch):
            return nn.Sequential(
                nn.ConvTranspose2d(in_ch,out_ch,4,2,1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )
        self.enc1=down(1,16)
        self.enc2=down(16,32)
        self.enc3=down(32,64)
        self.enc4=down(64,16)

        self.dec1=up(128,64)
        self.dec2=up(128,32)
        self.dec3=up(64,16)

        self.out_conv=nn.Conv2d(32,1,3,1,1)
        self.out_activation=nn.Sigmoid()

    def forward(self,x):
        e1=self.enc1(x)
        e2=self.enc2(e1)
        e3=self.enc3(e2)
        e4=self.enc4(e3)

        d1=self.dec1(e4)
        e3a=F.interpolate(e3,size=d1.size()[2:],mode="bilinear",align_corners=False)
        d1=torch.cat([d1,e3a],dim=1)

        d2=self.dec2(d1)
        e2a=F.interpolate(e2,size=d2.size()[2:],mode="bilinear",align_corners=False)
        d2=torch.cat([d2,e2a],dim=1)

        d3=self.dec3(d2)
        e1a=F.interpolate(e1,size=d3.size()[2:],mode="bilinear",align_corners=False)
        d3=torch.cat([d3,e1a],dim=1)

        out=self.out_conv(d3)
        out=self.out_activation(out)
        out=F.interpolate(out,size=x.shape[2:],mode="bilinear",align_corners=False)
        return out

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=UNetMask().to(device)
print("Model ready on",device)


Model ready on cpu


In [6]:
#METRICS(measures how much our model improves the audio quality using SNR)
def compute_snr(clean,test):
    clean=np.asarray(clean,dtype=np.float64)
    test=np.asarray(test,dtype=np.float64)

    min_len=min(len(clean),len(test))
    clean=clean[:min_len]
    test=test[:min_len]

    noise=clean-test
    signal_power=np.sum(clean**2)+1e-12
    noise_power=np.sum(noise**2)+1e-12

    return 10*np.log10(signal_power/noise_power)

def evaluate_snr(clean,noisy,enhanced):
    snr_noisy=compute_snr(clean,noisy)
    snr_enh=compute_snr(clean,enhanced)
    delta_snr=snr_enh-snr_noisy

    return snr_noisy,snr_enh,delta_snr